# Lizard Species Classification with Transfer Learning
## Using EfficientNetV2S on Google Colab (GPU)

In this notebook we will classify **7 lizard species** using **transfer learning** — the technique of reusing a neural network that was already trained on a huge dataset (ImageNet, ~14 million images) and adapting it to our smaller, more specific problem.

The species we need to classify are:
| Label | Class |
|-------|-------|
| 0 | Black spiny-tailed iguana |
| 1 | Brown anole |
| 2 | Cuban knight anole |
| 3 | Desert iguana |
| 4 | Green anole |
| 5 | Green iguana |
| 6 | Lesser Antillean iguana |

### Why Transfer Learning?
We only have ~1,300 training images — far too few to train a deep CNN from scratch without heavily overfitting. By starting from a network that already knows how to recognise edges, textures, and shapes from 14 million images, we only need to teach it the *difference* between our 7 lizard classes.

### Strategy — Three-Phase Progressive Unfreezing
1. **Phase 1** — freeze the entire backbone, train only the classification head we add on top
2. **Phase 2** — unfreeze the top 50 backbone layers, fine-tune with a low learning rate
3. **Phase 3** — unfreeze all layers, fine-tune the whole network with a very low learning rate

Each phase uses a progressively lower learning rate to avoid destroying the knowledge the backbone already has.

### Grad-CAM Visualisation
After training, we use **Gradient-weighted Class Activation Mapping (Grad-CAM)** to see *which parts of the image* the model is actually using to make its decision. This tells us whether the model is cheating (looking at the background colour, rocks, grass) or genuinely learning to recognise species-specific features.

---
## 0. Google Colab Setup

This notebook is designed to run on **Google Colab with a GPU runtime**.  
Training on a GPU is roughly 10–50× faster than on a CPU for deep learning tasks like this.

**To enable GPU in Colab:**  
`Runtime → Change runtime type → Hardware accelerator → T4 GPU → Save`

**To upload your data:**
1. Zip your `train/`, `test/`, and `test.csv` into a single file (e.g. `lizards.zip`)
2. In Colab, click the **folder icon** on the left panel
3. Drag and drop your zip file into session storage
4. Run the unzip cell below — it will extract the data into `/content/`

> **Note:** Session storage is temporary — if your Colab session disconnects, you will need to re-upload. For a permanent solution, mount Google Drive instead.

In [ ]:
import os

ON_COLAB = os.path.exists('/content')

if ON_COLAB:
    # Uncomment and adjust the filename to match your uploaded zip
    # !unzip -q /content/lizards.zip -d /content/
    print('Running on Google Colab')
    print('Make sure to unzip your data before continuing!')
else:
    print('Running locally — paths will point to the project data folder')

---
## 1. Install and Import Libraries

We install and import everything we need:
- **TensorFlow / Keras** — deep learning framework and pre-trained models
- **NumPy / Pandas** — numerical operations and CSV handling
- **Matplotlib / OpenCV** — visualisations and image manipulation
- **scikit-learn** — confusion matrix

In [ ]:
!pip install tensorflow numpy matplotlib scikit-learn opencv-python pillow -q
%pip install opencv-python opencv-contrib-python
%pip install pandas

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
import cv2
cv2.__version__
%matplotlib inline

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.regularizers import l2

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report

# Verify GPU availability
print('TensorFlow version:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPU devices found:', gpus)
if not gpus:
    print('WARNING: No GPU detected! Go to Runtime → Change runtime type → T4 GPU')

---
## 2. Configuration

All key settings are defined in one place so they are easy to adjust.

**Image size: 384×384**  
This is the *native* input resolution of EfficientNetV2S — the resolution it was originally trained with. Using this exact size means the model's spatial feature maps have the same scale as during ImageNet training, which gives us the best possible starting point for fine-tuning.

**Batch size: 32**  
A GPU can process many images in parallel. A batch of 32 images at 384×384 is a reasonable size that balances training speed with memory usage.

In [ ]:
# ── Data paths — adjust based on where you uploaded your data ─────────────────
if ON_COLAB:
    TRAIN_DIR = '/content/train'
    TEST_DIR  = '/content/test'
    TEST_CSV  = '/content/test.csv'
else:
    TRAIN_DIR = '../train'
    TEST_DIR  = '../test'
    TEST_CSV  = '../test.csv'
# ─────────────────────────────────────────────────────────────────────────────

IMG_SIZE    = (384, 384)   # EfficientNetV2S native resolution
BATCH_SIZE  = 32
NUM_CLASSES = 7
SEED        = 42

# Class labels — alphabetical order matches how Keras reads the folder names
CLASS_NAMES = [
    'Black_spiny_tailed_iguana',   # 0
    'Brown_anole',                 # 1
    'Cuban_knight_anole',          # 2
    'Desert_iguana',               # 3
    'Green_anole',                 # 4
    'Green_iguana',                # 5
    'Lesser_Antillean_iguana',     # 6
]

# Short display labels for plots
SHORT_NAMES = [c.replace('_', ' ') for c in CLASS_NAMES]

print('Configuration:')
print(f'  Image size  : {IMG_SIZE}')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Classes     : {NUM_CLASSES}')
print(f'  Train dir   : {TRAIN_DIR}')

---
## 3. Exploratory Data Analysis (EDA)

Before training any model, it is important to *understand the data*. We want to answer:

1. **How many images do we have per class?** — Are the classes balanced?
2. **What do the images look like?** — Are there noisy or faulty images?
3. **How large are the images?** — We need to resize them all to the same size for the network.
4. **Do the species differ in colour?** — If one class is always green and another is always brown, the model might just learn the colour as a shortcut instead of the actual shape. We will use **Grad-CAM** later to verify whether this is happening.

> Faulty or corrupted images exist in this dataset. We use `try/except` blocks throughout to handle them gracefully.

In [ ]:
# ── 3.1  Class distribution ───────────────────────────────────────────────────
class_counts = {}
for cls in CLASS_NAMES:
    imgs = glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg'))
    class_counts[cls] = len(imgs)

total_images = sum(class_counts.values())

fig, ax = plt.subplots(figsize=(13, 4))
bars = ax.bar(
    [n.replace('_', '\n') for n in CLASS_NAMES],
    class_counts.values(),
    color=plt.cm.tab10(np.linspace(0, 0.8, NUM_CLASSES)),
    edgecolor='white', linewidth=1.5
)
for bar, (cls, count) in zip(bars, class_counts.items()):
    pct = count / total_images * 100
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
            f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=8.5)

ax.set_title('Training Images per Class', fontsize=14, pad=15)
ax.set_ylabel('Number of images')
ax.set_ylim(0, max(class_counts.values()) * 1.18)
ax.axhline(total_images / NUM_CLASSES, color='grey', linestyle='--', linewidth=1)
ax.text(NUM_CLASSES - 0.4, total_images / NUM_CLASSES + 1,
        'class average', color='grey', fontsize=8, ha='right')
plt.tight_layout()
plt.show()

print(f'Total training images : {total_images}')
print(f'Average per class     : {total_images / NUM_CLASSES:.1f}')
print(f'Min / Max             : {min(class_counts.values())} / {max(class_counts.values())}')

The dataset is **well balanced** — all classes have a similar number of images. This is ideal for training, as it means the model does not get more examples of one class than another and therefore cannot cheat by always predicting the most common class.

In [ ]:
# ── 3.2  Image size distribution ──────────────────────────────────────────────
# Sample up to 40 images per class to build a scatter plot of width vs height.
# This tells us how variable the image sizes are, and confirms we need resizing.

widths, heights, cls_indices = [], [], []
colours = plt.cm.tab10(np.linspace(0, 0.8, NUM_CLASSES))

for i, cls in enumerate(CLASS_NAMES):
    sample = sorted(glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')))[:40]
    for img_path in sample:
        try:
            with Image.open(img_path) as im:
                w, h = im.size
            widths.append(w); heights.append(h); cls_indices.append(i)
        except Exception:
            pass   # skip faulty images

fig, ax = plt.subplots(figsize=(9, 6))
for i, cls in enumerate(CLASS_NAMES):
    mask = [j == i for j in cls_indices]
    ax.scatter(
        [w for w, m in zip(widths,  mask) if m],
        [h for h, m in zip(heights, mask) if m],
        label=SHORT_NAMES[i], alpha=0.55, s=35, color=colours[i]
    )

# Mark the target size
ax.axvline(IMG_SIZE[0], color='red', linestyle='--', linewidth=1.5, label=f'Target ({IMG_SIZE[0]}px)')
ax.axhline(IMG_SIZE[1], color='red', linestyle='--', linewidth=1.5)

ax.set_xlabel('Original width (px)', fontsize=11)
ax.set_ylabel('Original height (px)', fontsize=11)
ax.set_title('Image Size Distribution per Class (sample of 40 per class)', fontsize=12)
ax.legend(fontsize=7.5, loc='lower right', ncol=2)
plt.tight_layout()
plt.show()

print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f} px')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f} px')
print(f'\nAll images will be resized to {IMG_SIZE} before entering the network.')

The images are **high resolution but inconsistent in size** — some are portrait, some landscape, and the dimensions vary widely. The network needs a fixed input size, so all images are resized to **384×384** during loading. Note that some detail is lost when a 2048px image is shrunk to 384px, but 384×384 is still significantly more detail than the 224×224 used by many older models.

In [ ]:
# ── 3.3  Sample images — 3 per class ─────────────────────────────────────────
# Visual inspection is always a good idea before training.
# Look for: variety of poses, backgrounds, lighting, and any obviously corrupted images.

SAMPLES = 3
fig, axes = plt.subplots(NUM_CLASSES, SAMPLES, figsize=(SAMPLES * 3.5, NUM_CLASSES * 3))

for row, cls in enumerate(CLASS_NAMES):
    sample_imgs = sorted(glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')))[:SAMPLES]
    for col in range(SAMPLES):
        ax = axes[row, col]
        if col < len(sample_imgs):
            try:
                img = tf.keras.utils.load_img(sample_imgs[col], target_size=(200, 200))
                ax.imshow(img)
            except Exception:
                ax.set_facecolor('#ddd')
                ax.text(0.5, 0.5, 'faulty', ha='center', va='center',
                        transform=ax.transAxes, color='red')
        ax.axis('off')
        if col == 0:
            ax.set_title(SHORT_NAMES[row], fontsize=9, loc='left', pad=3)

plt.suptitle('Sample Images per Class (3 per row)', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.4  Average dominant colour per class ─────────────────────────────────────
#
# We compute the mean RGB colour across all pixels of a sample of images
# for each class. If the average colours are very different, the model might
# learn to simply classify by dominant colour — a shortcut that would break
# on images with unusual lighting or different backgrounds.
#
# We use Grad-CAM later to check whether this is actually happening.

avg_rgb = []
std_rgb = []

for cls in CLASS_NAMES:
    sample = sorted(glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')))[:60]
    all_pixels = []
    for img_path in sample:
        try:
            img = tf.keras.utils.load_img(img_path, target_size=(64, 64))
            arr = tf.keras.utils.img_to_array(img) / 255.0
            all_pixels.append(arr.reshape(-1, 3))
        except Exception:
            pass
    if all_pixels:
        stacked = np.concatenate(all_pixels, axis=0)
        avg_rgb.append(np.mean(stacked, axis=0))
        std_rgb.append(np.std(stacked, axis=0))
    else:
        avg_rgb.append(np.array([0.5, 0.5, 0.5]))
        std_rgb.append(np.zeros(3))

# Plot colour swatches
fig, axes = plt.subplots(1, 2, figsize=(14, 3))

# Left: colour swatch per class
ax = axes[0]
for i, (cls, colour) in enumerate(zip(CLASS_NAMES, avg_rgb)):
    ax.add_patch(mpatches.Rectangle((i, 0), 1, 1, color=np.clip(colour, 0, 1)))
    ax.text(i + 0.5, -0.08, SHORT_NAMES[i].replace(' ', '\n'),
            ha='center', va='top', fontsize=7)
ax.set_xlim(0, NUM_CLASSES)
ax.set_ylim(-0.7, 1)
ax.axis('off')
ax.set_title('Average Dominant Colour per Class', fontsize=11, pad=8)

# Right: R/G/B channel breakdown
ax2 = axes[1]
x = np.arange(NUM_CLASSES)
w = 0.25
channel_colours = ['#e74c3c', '#2ecc71', '#3498db']
channel_labels  = ['Red channel', 'Green channel', 'Blue channel']
for ci in range(3):
    vals = [avg_rgb[i][ci] for i in range(NUM_CLASSES)]
    ax2.bar(x + ci * w, vals, w, label=channel_labels[ci],
            color=channel_colours[ci], alpha=0.8)
ax2.set_xticks(x + w)
ax2.set_xticklabels([n.split(' ')[0] for n in SHORT_NAMES], rotation=30, ha='right', fontsize=8)
ax2.set_ylabel('Average pixel intensity [0-1]')
ax2.set_title('RGB Channel Breakdown per Class', fontsize=11)
ax2.legend(fontsize=8)
ax2.set_ylim(0, 0.7)

plt.tight_layout()
plt.show()

print('Observation: if some classes have very different dominant colours,')
print('the model may use colour as a shortcut. Grad-CAM will help detect this.')

---
## 4. Data Loading

We use `image_dataset_from_directory`, which scans the folder structure and automatically assigns labels based on subfolder names. This is the modern preferred approach in TensorFlow 2.x — it builds an efficient `tf.data.Dataset` pipeline that loads images on-the-fly during training.

The dataset is split **80% training / 20% validation**, using the same random seed so the split is reproducible.

After loading, we apply:
- **`.cache()`** — stores the entire dataset in memory after the first epoch, so subsequent epochs are much faster
- **`.shuffle()`** — randomises the order of images each epoch, which helps generalisation
- **`.prefetch()`** — prepares the next batch while the GPU is processing the current one

In [ ]:
train_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',         # integer labels (0-6); sparse_categorical_crossentropy
)

val_ds = image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
)

print('Class order (must match CLASS_NAMES above):', train_ds.class_names)
print(f'Train batches: {len(train_ds)}  ({len(train_ds) * BATCH_SIZE} images approx.)')
print(f'Val   batches: {len(val_ds)}   ({len(val_ds)   * BATCH_SIZE} images approx.)')

# Performance optimisations
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000, seed=SEED).prefetch(AUTOTUNE)
val_ds   = val_ds.cache().prefetch(AUTOTUNE)

---
## 5. Data Augmentation

**Data augmentation** artificially increases the effective size of our training set by applying random transformations to each image every time it is loaded. The label does not change — a randomly rotated lizard is still the same lizard. But the random variation forces the model to learn *invariant* features (shape, texture) rather than memorising specific image files.

This is especially important for our dataset because:
- We only have ~1,300 images (~185 per class) — a very small dataset for deep learning
- Some images are faulty or corrupted — augmentation makes the model more robust to noise

**Why these augmentations specifically?**

| Augmentation | Reason |
|---|---|
| `RandomFlip` | Lizards face both directions in photos |
| `RandomRotation` | Camera angle varies |
| `RandomZoom` | Lizard may fill the frame or be in the background |
| `RandomTranslation` | Lizard is rarely perfectly centred |
| `RandomBrightness` | Outdoor lighting conditions vary widely |
| `RandomContrast` | Shadow, overexposure, different cameras |

The last two (`RandomBrightness` + `RandomContrast`) also specifically discourage the model from using dominant colour as its main classification feature — a potential shortcut we saw in the EDA.

In [ ]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
    layers.RandomTranslation(0.1, 0.1),
    layers.RandomBrightness(0.2),
    layers.RandomContrast(0.2),
], name='augmentation')

In [ ]:
# Visualise the effect of augmentation on a sample batch
# Each row shows the same original image with different random augmentations applied

for images, labels in train_ds.take(1):
    sample_image = images[0:1]                        # pick the first image
    sample_label = CLASS_NAMES[int(labels[0])]

    n_versions = 8
    fig, axes = plt.subplots(2, 4, figsize=(14, 6))
    for i, ax in enumerate(axes.flat):
        aug = data_augmentation(sample_image, training=True)[0].numpy().astype('uint8')
        ax.imshow(aug)
        ax.set_title(f'Augmented #{i+1}', fontsize=9)
        ax.axis('off')
    plt.suptitle(f'Data Augmentation Examples — Class: {sample_label.replace("_", " ")}',
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.show()
    break

---
## 6. Model Architecture — EfficientNetV2S with Transfer Learning

### Why EfficientNetV2S?

We need a model that:
1. Is available directly in `tf.keras.applications` (no extra libraries)
2. Was pre-trained on ImageNet at high resolution (so it already knows what textures and shapes look like)
3. Is accurate but not so large that fine-tuning takes days

**EfficientNetV2S** ticks all three boxes. It is significantly more accurate than older choices like MobileNetV2 or ResNet50, while remaining trainable on a single T4 GPU in a reasonable time.

### Key setting: `include_preprocessing=True`

EfficientNetV2S was trained with a specific pixel normalisation. Setting `include_preprocessing=True` means the model handles this internally — we can pass raw `[0, 255]` pixel values directly without adding our own rescaling layer.

### Classification Head

We replace the original ImageNet top (1000-class Dense layer) with our own:
- **GlobalAveragePooling2D** — collapses the spatial feature maps into a single vector
- **Dropout(0.5)** — heavy dropout to combat both overfitting and the noisy/faulty images in the dataset
- **Dense(256, relu)** — intermediate layer to learn lizard-specific combinations of features
- **Dropout(0.3)** — second dropout for additional regularisation
- **Dense(7, softmax)** — final layer: one output neuron per class, softmax ensures probabilities sum to 1

In [ ]:
# Load the EfficientNetV2S backbone — all weights from ImageNet, no top layer
base_model = EfficientNetV2S(
    input_shape=IMG_SIZE + (3,),
    include_top=False,           # we add our own classification head below
    weights='imagenet',          # pretrained weights (downloads ~90 MB)
    include_preprocessing=True,  # handles pixel normalisation internally
)
base_model.trainable = False     # freeze all backbone layers for Phase 1

print(f'Backbone : {base_model.name}')
print(f'Layers   : {len(base_model.layers)}')
print(f'Parameters: {base_model.count_params():,}')

In [ ]:
# Build the full model using the Functional API
# (Functional API is preferred over Sequential when we need fine-grained control,
#  e.g. for Grad-CAM visualisations later)

inputs  = tf.keras.Input(shape=IMG_SIZE + (3,), name='input_image')
x       = data_augmentation(inputs)                            # random augmentation during training
x       = base_model(x, training=False)                        # frozen backbone
x       = GlobalAveragePooling2D(name='gap')(x)                # (batch, 1280)
x       = Dropout(0.5, name='dropout_1')(x)                    # strong dropout — noisy dataset
x       = Dense(256, activation='relu',
                kernel_regularizer=l2(1e-4),
                name='dense_1')(x)                             # intermediate head
x       = Dropout(0.3, name='dropout_2')(x)
outputs = Dense(NUM_CLASSES, activation='softmax',
                name='predictions')(x)                         # 7-class output

model = Model(inputs, outputs, name='lizard_efficientnetv2s')
model.summary()

In [ ]:
# Show the last few backbone layers — the last activation layer is used for Grad-CAM
print('Last 6 backbone layers (for Grad-CAM reference):')
for layer in base_model.layers[-6:]:
    out_shape = getattr(layer, 'output', None)
    print(f'  {layer.name:<50} {type(layer).__name__}')

---
## 7. Phase 1 — Train the Classification Head (Backbone Frozen)

In the first phase, the **entire EfficientNetV2S backbone is frozen** — its 20+ million parameters will not be updated. Only the small head we added (~330k parameters) is trained.

**Why start like this?**  
The backbone's weights are already well-optimised for recognising visual features. If we immediately tried to train the entire network with random weights in the head, the large gradient signals from the untrained head would destroy the carefully learned backbone weights. By freezing the backbone first, we safely warm up the head.

**Learning rate: `1e-3`**  
Relatively high, since the head starts from random weights and needs to learn quickly.

**Callbacks:**
- `EarlyStopping` — stops training if validation accuracy does not improve for 7 epochs, and restores the best weights found
- `ReduceLROnPlateau` — halves the learning rate when validation loss plateaus, helping squeeze out extra performance
- `ModelCheckpoint` — saves the best model weights to disk

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p1 = [
    EarlyStopping(monitor='val_accuracy', patience=7,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3,
                      min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_phase1.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Phase 1 — training HEAD only')
print(f'Trainable parameters : {trainable_params:,}')
print(f'Frozen parameters    : {base_model.count_params():,}')
print()

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,              # EarlyStopping will cut this short once learning plateaus
    callbacks=callbacks_p1,
)

---
## 8. Phase 2 — Fine-Tune the Top Backbone Layers

Now we **unfreeze the last 50 layers** of the backbone. These are the deeper, high-level layers that learn complex patterns (species-specific textures, colour gradients, body shapes). The earlier layers (which learn basic edges and low-level textures) stay frozen because those are already excellent.

**Learning rate: `1e-4`** — 10× lower than Phase 1.  
We use a very gentle update step to nudge the pre-trained backbone weights slightly towards lizard-specific features, without destroying what they already know.

> **Important:** we must call `model.compile()` again after changing which layers are trainable, for the change to take effect.

In [ ]:
# Unfreeze the last 50 layers of the backbone
base_model.trainable = True
UNFREEZE_FROM = len(base_model.layers) - 50
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

frozen_count    = sum(1 for l in base_model.layers if not l.trainable)
unfrozen_count  = sum(1 for l in base_model.layers if     l.trainable)
print(f'Frozen backbone layers  : {frozen_count}')
print(f'Unfrozen backbone layers: {unfrozen_count}')

# Must recompile after changing trainability
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p2 = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                      min_lr=1e-8, verbose=1),
    ModelCheckpoint('best_phase2.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'\nPhase 2 — fine-tuning TOP 50 backbone layers')
print(f'Trainable parameters : {trainable_params:,}')
print()

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks_p2,
)

---
## 9. Phase 3 — Full Fine-Tune (All Layers)

In the final phase, **all backbone layers are unfrozen**. Every parameter in the entire network is updated together. This phase makes the smallest changes — the learning rate `1e-5` is 100× lower than Phase 1 — and relies on the model already having a good starting point from Phases 1 and 2.

Full fine-tuning allows the model to adapt even the early, low-level feature detectors (edges, colours) slightly towards the lizard domain, which can squeeze out the last few percentage points of accuracy.

In [ ]:
# Unfreeze all backbone layers
base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p3 = [
    EarlyStopping(monitor='val_accuracy', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=4,
                      min_lr=1e-9, verbose=1),
    ModelCheckpoint('best_phase3.keras', monitor='val_accuracy',
                    save_best_only=True, verbose=0),
]

trainable_params = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Phase 3 — FULL fine-tune (all layers unfrozen)')
print(f'Trainable parameters : {trainable_params:,}')
print()

history3 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks_p3,
)

---
## 10. Training Curves

The training curve shows how accuracy and loss changed over all three phases combined. The vertical dashed lines mark where Phase 2 and Phase 3 began.

**What to look for:**
- `val_accuracy` (validation) should generally follow `accuracy` (training). If training accuracy is much higher, the model may be overfitting
- Each phase boundary should ideally show a further improvement — confirmation that progressive unfreezing helped
- `ReduceLROnPlateau` events appear as sudden drops in the loss curves

In [ ]:
def combine_histories(*histories):
    combined = {}
    for h in histories:
        for key, vals in h.history.items():
            combined.setdefault(key, []).extend(vals)
    return combined


def plot_training_history(histories, phase_labels=None):
    combined     = combine_histories(*histories)
    lengths      = [len(h.history['accuracy']) for h in histories]
    phase_labels = phase_labels or [f'Phase {i+1}' for i in range(len(histories))]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    colours = ['#2ecc71', '#e74c3c']
    labels  = ['Train', 'Validation']

    for col, (metric, ax) in enumerate(zip(['accuracy', 'loss'], axes)):
        train_vals = combined[metric]
        val_vals   = combined[f'val_{metric}']
        epochs     = range(len(train_vals))

        ax.plot(epochs, train_vals, color=colours[0], linewidth=2, label='Train')
        ax.plot(epochs, val_vals,   color=colours[1], linewidth=2, label='Validation')

        # Phase boundaries
        boundary = 0
        for pi, length in enumerate(lengths[:-1]):
            boundary += length
            ax.axvline(boundary - 0.5, color='#95a5a6', linestyle='--', linewidth=1.2)
            ax.text(boundary, ax.get_ylim()[0] + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.02,
                    f' {phase_labels[pi+1]}', fontsize=8.5, color='#636e72')

        # Highlight best val point
        best_fn = np.argmax if metric == 'accuracy' else np.argmin
        best_ep = best_fn(val_vals)
        ax.scatter(best_ep, val_vals[best_ep], color='gold', s=80, zorder=5,
                   edgecolors='black', linewidth=0.8, label=f'Best val ({val_vals[best_ep]:.3f})')

        ax.set_title(metric.capitalize(), fontsize=13)
        ax.set_xlabel('Epoch')
        ax.set_ylabel(metric.capitalize())
        ax.legend(fontsize=9)
        ax.grid(alpha=0.3)

    plt.suptitle('Training History — All 3 Phases', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()


plot_training_history(
    [history1, history2, history3],
    phase_labels=['Phase 1\n(head only)', 'Phase 2\n(top 50)', 'Phase 3\n(full)']
)

p_lens = [len(h.history['accuracy']) for h in [history1, history2, history3]]
print(f'Epochs per phase: Phase 1={p_lens[0]}, Phase 2={p_lens[1]}, Phase 3={p_lens[2]}')

---
## 11. Model Evaluation

We evaluate the final model on the **validation set** — images the model has never seen during training.

### Confusion Matrix

A confusion matrix shows, for each true class (rows), how many images were predicted as each class (columns). The diagonal shows correct predictions.

**What to look for:**
- Large numbers on the diagonal → the model classifies those classes well
- Off-diagonal numbers → specific misclassifications (e.g. Green anole confused with Brown anole)
- Systematically low row values → the model struggles with a particular species

In [ ]:
# Collect all validation predictions
y_true, y_pred = [], []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
short_labels = [n.replace('_', '\n') for n in CLASS_NAMES]

fig, ax = plt.subplots(figsize=(12, 9))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=short_labels)
disp.plot(ax=ax, xticks_rotation=0, colorbar=True, cmap='Blues')
ax.set_title('Validation Confusion Matrix', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

# Overall accuracy
val_acc = np.sum(y_true == y_pred) / len(y_true)
print(f'Overall validation accuracy: {val_acc:.4f}  ({val_acc*100:.2f}%)')

In [ ]:
# Per-class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)

colours_acc = ['#27ae60' if a >= 0.7 else '#f39c12' if a >= 0.5 else '#e74c3c'
               for a in per_class_acc]

plt.figure(figsize=(12, 4))
bars = plt.bar(
    [n.replace('_', '\n') for n in CLASS_NAMES],
    per_class_acc,
    color=colours_acc, edgecolor='white', linewidth=1.5
)
for bar, acc in zip(bars, per_class_acc):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{acc:.0%}', ha='center', va='bottom', fontsize=9)

plt.axhline(val_acc, color='navy', linestyle='--', linewidth=1.5)
plt.text(NUM_CLASSES - 0.3, val_acc + 0.01, f'Overall {val_acc:.0%}',
         ha='right', color='navy', fontsize=9)

green_patch  = mpatches.Patch(color='#27ae60',  label='≥ 70%')
orange_patch = mpatches.Patch(color='#f39c12', label='50–70%')
red_patch    = mpatches.Patch(color='#e74c3c',  label='< 50%')
plt.legend(handles=[green_patch, orange_patch, red_patch], fontsize=9)

plt.ylim(0, 1.12)
plt.ylabel('Accuracy')
plt.title('Per-Class Validation Accuracy', fontsize=13)
plt.tight_layout()
plt.show()

# Detailed classification report
print('\nDetailed classification report:\n')
print(classification_report(y_true, y_pred, target_names=SHORT_NAMES))

---
## 12. Grad-CAM — What Is the Model Actually Looking At?

### What is Grad-CAM?

**Gradient-weighted Class Activation Mapping (Grad-CAM)** answers the question: *which pixels in the input image are most responsible for the model's prediction?*

**How it works (simplified):**
1. We do a forward pass through the backbone and record the **activation maps** at the last convolutional layer — these are the high-level feature maps the backbone produces
2. We compute the **gradient** of the predicted class score with respect to those activation maps — this tells us which features (channels) were most important for predicting that class
3. We **weight each activation map** by its gradient and average them together — this gives us a single heatmap showing which spatial locations mattered most
4. We overlay this heatmap on the original image using a colour map (red = high importance, blue = low importance)

**Why we use `tape.watch(conv_outputs)` before computing the score:**  
TensorFlow's `GradientTape` only automatically tracks gradients for trainable Variables. Since we want gradients with respect to a *tensor* (the backbone's output), we must explicitly call `tape.watch()` to tell the tape to record operations on it. We do this *before* computing the class score so that the path from the tensor to the score is fully recorded.

### What to look for in the heatmaps

| Observation | Interpretation |
|---|---|
| Red/yellow on the lizard's body or head | ✅ Model is learning the right features |
| Red/yellow on the background (rocks, grass, sky) | ⚠️ Model is using background as a shortcut |
| Red/yellow on a single colour patch | ⚠️ Model is relying on dominant colour (shortcut) |

If you see shortcuts, the fix is more aggressive colour/brightness augmentation, or adding a saliency-based crop to remove uninformative backgrounds.

In [ ]:
def compute_gradcam(img_array, class_idx=None):
    """
    Compute a Grad-CAM heatmap for a single image.

    Parameters
    ----------
    img_array : np.ndarray, shape (H, W, 3), pixel values in [0, 255]
    class_idx : int or None
        Which class to generate the heatmap for.
        If None, uses the model's top predicted class.

    Returns
    -------
    heatmap   : np.ndarray, shape (h, w), values in [0, 1]
    probs     : np.ndarray, shape (7,)   — class probabilities
    class_idx : int                      — the class used for the heatmap
    """
    img_tensor = tf.cast(img_array[np.newaxis, ...], tf.float32)  # (1, H, W, 3)

    with tf.GradientTape() as tape:
        # Step 1: forward pass through the backbone (no augmentation at inference)
        conv_outputs = base_model(img_tensor, training=False)     # (1, h, w, 1280)

        # Step 2: watch conv_outputs so gradients w.r.t. it are tracked
        tape.watch(conv_outputs)

        # Step 3: replay the head manually (no Dropout at inference)
        pooled = tf.reduce_mean(conv_outputs, axis=[1, 2])        # GAP: (1, 1280)
        d1     = model.get_layer('dense_1')                       # intermediate Dense
        x_head = tf.nn.relu(pooled @ d1.kernel + d1.bias)        # (1, 256)
        d2     = model.get_layer('predictions')                   # output Dense
        logits = x_head @ d2.kernel + d2.bias                    # (1, 7)
        probs  = tf.nn.softmax(logits)                           # (1, 7)

        # Step 4: select the class score to differentiate
        if class_idx is None:
            class_idx = int(tf.argmax(probs[0]))
        score = probs[:, class_idx]                              # scalar

    # Step 5: gradients of score w.r.t. conv_outputs
    grads        = tape.gradient(score, conv_outputs)            # (1, h, w, 1280)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))         # (1280,)  — importance per channel

    # Step 6: weight activation maps by their gradient importance, then sum
    heatmap = tf.reduce_sum(conv_outputs[0] * pooled_grads, axis=-1)  # (h, w)
    heatmap = tf.maximum(heatmap, 0)                             # ReLU — ignore negative contributions
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)         # normalise to [0, 1]

    return heatmap.numpy(), probs.numpy()[0], class_idx


def show_gradcam(img_path, class_idx=None, title=None):
    """Load an image, compute Grad-CAM, and show original / heatmap / overlay."""
    img       = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
    img_array = tf.keras.utils.img_to_array(img)   # shape (H, W, 3), [0, 255]

    heatmap, probs, pred_class = compute_gradcam(img_array, class_idx)

    # Resize the low-resolution heatmap back to full image size
    heatmap_resized = cv2.resize(heatmap, IMG_SIZE)

    # Apply jet colour map: blue (low) → green → red (high)
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)   # BGR → RGB

    # Blend original image and heatmap
    overlay = np.clip(0.55 * img_array + 0.45 * heatmap_colored, 0, 255).astype(np.uint8)

    # Display
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    axes[0].imshow(img_array.astype(np.uint8))
    axes[0].set_title('Original image', fontsize=11)
    axes[0].axis('off')

    axes[1].imshow(heatmap, cmap='jet')
    axes[1].set_title('Grad-CAM heatmap\n(red = high attention, blue = low attention)', fontsize=10)
    axes[1].axis('off')

    top3 = np.argsort(probs)[::-1][:3]
    top3_text = '\n'.join([f'{CLASS_NAMES[j].replace("_"," ")}: {probs[j]:.1%}' for j in top3])
    axes[2].imshow(overlay)
    axes[2].set_title(f'Overlay\nTop-3 predictions:\n{top3_text}', fontsize=9)
    axes[2].axis('off')

    suptitle = title or os.path.basename(img_path)
    plt.suptitle(suptitle, fontsize=10, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# Show Grad-CAM for the first correctly loaded image from each class
print('Generating Grad-CAM visualisations — one image per class\n')
print('Check: is the heatmap on the LIZARD (good) or on the BACKGROUND/COLOUR (bad)?\n')

for cls in CLASS_NAMES:
    imgs = sorted(glob.glob(os.path.join(TRAIN_DIR, cls, '*.jpg')))
    shown = False
    for img_path in imgs:
        try:
            show_gradcam(img_path, title=f'Class: {cls.replace("_", " ")}')
            shown = True
            break
        except Exception as e:
            continue   # skip faulty images
    if not shown:
        print(f'Could not load any image for class: {cls}')

In [ ]:
# Show Grad-CAM on validation images that were MISCLASSIFIED
# This helps understand what confused the model
print('Grad-CAM on misclassified validation images\n')

shown = 0
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    pred_labels = np.argmax(preds, axis=1)
    for i in range(len(labels)):
        true_label = int(labels[i])
        pred_label = int(pred_labels[i])
        if true_label != pred_label:
            img_array = images[i].numpy()
            heatmap, probs, _ = compute_gradcam(img_array, class_idx=pred_label)

            heatmap_resized = cv2.resize(heatmap, IMG_SIZE)
            heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
            heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB)
            overlay = np.clip(0.55 * img_array + 0.45 * heatmap_colored, 0, 255).astype(np.uint8)

            fig, axes = plt.subplots(1, 3, figsize=(16, 5))
            axes[0].imshow(img_array.astype(np.uint8))
            axes[0].set_title(f'True: {CLASS_NAMES[true_label].replace("_"," ")}', fontsize=10)
            axes[0].axis('off')
            axes[1].imshow(heatmap, cmap='jet')
            axes[1].set_title('Grad-CAM heatmap', fontsize=10)
            axes[1].axis('off')
            axes[2].imshow(overlay)
            axes[2].set_title(
                f'Predicted: {CLASS_NAMES[pred_label].replace("_"," ")} ({probs[pred_label]:.1%})',
                fontsize=10
            )
            axes[2].axis('off')
            plt.suptitle('Misclassified Example', fontsize=11, color='crimson', y=1.01)
            plt.tight_layout()
            plt.show()

            shown += 1
            if shown >= 4:   # show at most 4 misclassified examples
                break
    if shown >= 4:
        break

---
## 13. Generate Kaggle Submission

We load each test image, predict the class with the trained model, and save the results to a CSV file. The CSV is then uploaded to Kaggle as our submission.

Faulty test images are handled with a `try/except` — any image that cannot be loaded falls back to class `0`.

In [ ]:
test_df  = pd.read_csv(TEST_CSV)
test_ids = test_df['id'].tolist()

predictions = []
for img_id in test_ids:
    img_path = os.path.join(TEST_DIR, f'{img_id}.jpg')
    try:
        img       = tf.keras.utils.load_img(img_path, target_size=IMG_SIZE)
        img_array = tf.keras.utils.img_to_array(img)
        img_array = np.expand_dims(img_array, axis=0)       # add batch dimension
        pred      = model.predict(img_array, verbose=0)
        predictions.append(int(np.argmax(pred, axis=1)[0]))
    except Exception:
        predictions.append(0)   # fallback for faulty test images

out_path   = 'submission_efficientnetv2s.csv'
submission = pd.DataFrame({'id': test_ids, 'label': predictions})
submission.to_csv(out_path, index=False)

print(f'Saved {len(submission)} predictions → {out_path}')
print(f'Label distribution in submission:')
print(submission['label'].value_counts().sort_index()
      .rename(index=dict(enumerate(CLASS_NAMES))).to_string())

submission.head(10)